In [1]:
import pandas as pd

In [2]:
# 定义数据集名称和列名：
datasets = ["LUAD", "LUSC", "BLCA", "BRCA", "KIRC", "LIHC"]
dataframe_columns = ["Dataset", "#Genes", "#Samples", "All Samples", "Imbalance Ratio"]

# 统计信息存储：
statistical_information = []

for dataset in datasets:
    # 读取数据：
    csv_file = f"./TCGA_GDC/dataset/{dataset}_TPM.csv"
    df = pd.read_csv(csv_file, index_col=0)

    # 统计 0 和 1 的数量：
    counts = df["label"].value_counts()
    # 获取 0 和 1 的数量：
    count_0 = counts.get(0, default=0)  # 如果列中没有 0，默认返回 0
    count_1 = counts.get(1, default=0)  # 如果列中没有 1，默认返回 0

    # 计算不平衡比例：
    imbalance_ratio = count_1 / count_0 if count_0 != 0 else float("inf")

    # 统计基因和样本数量：
    samples_num, genes_num = df.drop(columns=["label"]).shape

    # 添加统计信息：
    statistical_information.append([
        f"TCGA-{dataset}",
        genes_num,
        f"Cancer({count_1}) : Normal({count_0})",
        samples_num,
        imbalance_ratio
    ])

result_df = pd.DataFrame(statistical_information, columns=dataframe_columns)
result_df.index = list(range(1, len(result_df) + 1))
result_df

,Dataset,#Genes,#Samples,All Samples,Imbalance Ratio
1,TCGA-LUAD,17604,Cancer(513) : Normal(58),571,8.844828
2,TCGA-LUSC,17868,Cancer(496) : Normal(51),547,9.725490
3,TCGA-BLCA,17401,Cancer(403) : Normal(19),422,21.210526
4,TCGA-BRCA,17601,Cancer(1081) : Normal(99),1180,10.919192
5,TCGA-KIRC,17719,Cancer(529) : Normal(72),601,7.347222
6,TCGA-LIHC,16783,Cancer(369) : Normal(50),419,7.380000


In [ ]:
import json
from collections import Counter

import numpy as np
import pandas as pd
import seaborn as sns

import matplotlib as mpl
from matplotlib import ticker
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm


with open(f"./data/GDC_Genes_20908_2023-12-31.json", "r") as f:
    data1 = json.load(f)

print(f"字典列表长度：{len(data1)}")

# 提取生物型和染色体区带进行分析：
biotypes = [entry["biotype"] for entry in data1]
cytobands = [band for entry in data1 for band in entry["cytoband"]]

# 计算每种生物型的出现次数：
biotype_counts = Counter(biotypes)
# 将 Counter 对象转换为字典：
biotype_counts_dict = dict(biotype_counts)
# 按值排序字典并生成新的字典：
sorted_biotype_counts = sorted(
    biotype_counts_dict.items(), key=lambda x: x[1], reverse=True
)
gene_types = [k for k, _ in sorted_biotype_counts]
num = [v for _, v in sorted_biotype_counts]
# 按数量从大到小排序：
sorted_indices = sorted(range(len(num)), key=lambda i: num[i], reverse=True)
gene_types_sorted = [gene_types[i] for i in sorted_indices]
counts_sorted = [num[i] for i in sorted_indices]

# 将 Matplotlib 的参数设置（rcParams）恢复为默认值：
mpl.rcParams.update(mpl.rcParamsDefault)
font_latex1 = fm.FontProperties(
    fname="./fonts/Times New Roman.ttf", style="italic", size=17, weight="bold"
)
font_latex2 = fm.FontProperties(
    fname="./fonts/Times New Roman.ttf", style="italic", size=16, weight="bold"
)
# 将指定的字体文件添加到 matplotlib 的字体管理器中。这样，matplotlib 就可以使用这个字体文件来渲染图形中的文本。
fm.fontManager.addfont("./fonts/Times New Roman.ttf")
# 设置 matplotlib 的全局参数，将默认的无衬线字体（sans-serif）设置为 'Latin Modern Math'。这意味着在绘制图形时，如果没有指定其他字体，matplotlib 将使用 'Latin Modern Math' 字体来渲染文本。
prop = font_latex1
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = prop.get_name()
# 为了避免在使用某些字体时出现负号显示问题，我们需要设置 Matplotlib 的参数，使其在绘制负数时不使用 Unicode 的负号字符。
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.size"] = 16
# 创建图形和轴：
fig, ax = plt.subplots(figsize=(13, 9), dpi=600)

# 绘制水平条形图：
bars = plt.barh(gene_types_sorted, counts_sorted, color="#0052D9")

# 坐标轴刻度数值用 Latin Modern Math 字体：
labels = ax.get_xticklabels() + ax.get_yticklabels()
[label.set_fontproperties(font_latex2) for label in labels]
[label.set_color("#000000") for label in labels]
# 设置坐标轴刻度：
plt.tick_params(axis="x", direction="out", labelsize=16, length=4.6, width=1.2)
plt.tick_params(axis="y", direction="out", labelsize=16, length=4.6, width=1.2)
# 设置 x 轴和 y 轴的标签、字体、刻度、刻度标签，以及坐标轴边界框中的间距。
plt.xlabel("Number of Genes", fontproperties=font_latex1, labelpad=9)
ax.set_xlim(left=0, right=22500)
ax.set_xticks(np.arange(0, 22500.0000000001, step=2500))
# 仅在 x 轴上显示次刻度：
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
# 设置图表边框的线宽：
lw = 1.33
ax.spines["right"].set_linewidth(lw)
ax.spines["left"].set_linewidth(lw)
ax.spines["top"].set_linewidth(lw)
ax.spines["bottom"].set_linewidth(lw)

# 反转 y 轴，使得数量最多的在上方。
plt.gca().invert_yaxis()  # 从大到小

# 添加数值标签到条形图：
for bar in bars:
    width = bar.get_width()
    plt.text(
        width + max(counts_sorted) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{width}",
        va="center",
    )

# 显示网格，设置虚线，并调整透明度。
ax.grid(alpha=0.330, ls="--", which="major", color="#A9A9A9")
# 调整布局并保存图表：
plt.tight_layout()
plt.savefig("gene_types.png", dpi=600, transparent=True, bbox_inches="tight")

sorted_biotype_counts_dict = {k_: v_ for k_, v_ in sorted_biotype_counts}
print(sorted_biotype_counts_dict)
print(f"计数：{sum(counts_sorted)}")

# 计数每个染色体区带的出现：
cytoband_counts = Counter(cytobands)

# 显示结果：
print(biotype_counts)
print(cytoband_counts.most_common(10))  # 前 10 个最常见的染色体区带

# 将数据转换为 DataFrame 以便于操作：
df = pd.DataFrame(data1)

# 过滤 DataFrame 以获取感兴趣的特定生物型：
biotypes_of_interest = [
    "snoRNA",
    "transcribed_unprocessed_pseudogene",
    "processed_pseudogene",
    "protein_coding",
    "lncRNA",
    "miRNA",
]
filtered_df = df[df["biotype"].isin(biotypes_of_interest)]

# 分解染色体区带列表以每行一个染色体区带：
exploded_df = filtered_df.explode("cytoband").reset_index(drop=True)

# 按生物型和染色体区带分组，然后计数出现次数。
distribution_df = (
    exploded_df.groupby(["biotype", "cytoband"]).size().reset_index(name="counts")
)

# 对结果进行排序以提高可读性：
distribution_df = distribution_df.sort_values(
    by=["biotype", "counts"], ascending=[True, False]
)
print(distribution_df.head())  # 显示分布数据的前几行
distribution_df.to_csv(
    "distribution.csv", index=False, encoding="utf-8"
)  # 将分发数据保存到 CSV 文件

# 透视数据，使生物型作为列，染色体区带作为行。
pivot_df = distribution_df.pivot(
    index="cytoband", columns="biotype", values="counts"
).fillna(0)

# 绘制热图：
plt.figure(figsize=(14, 8))
sns.heatmap(pivot_df, cmap="YlGnBu", linewidths=0.5, cbar_kws={"label": "Count"})
plt.title("Distribution of Specific Gene Types Across Chromosomal Bands")
plt.ylabel("Chromosomal Band")
plt.xlabel("Gene Type")
plt.xticks(rotation=45)
plt.tight_layout()
# 将热图另存为 PNG 文件：
# plt.savefig("heatmap.png", dpi=600)

# 将用于热图的数据透视表导出到 CSV 文件：
export_path = "gene_type_distribution_across_cytobands.csv"
pivot_df.to_csv(export_path, index=True, encoding="utf-8")